In [1]:
%load_ext autoreload

In [2]:
from jsonargparse import CLI, set_parsing_settings

import os
import sys
import time
import warnings
from pathlib import Path
from pprint import pprint
from typing import Any, Dict, Iterator, List, Literal, Optional, Tuple, Union

import torch
from transformers import AutoTokenizer, TextStreamer

/capstor/scratch/cscs/jkirchen/daint_210_129_nightly_singleshot/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [33]:
# NOTE: hf version requires a conversion to exist at the target location

HF_MODEL_GPU = 0 # 1,2,3

# Example of local path debugging
# New Qwen3-4B-Instruct-2507 trained on bos only metamath, max k 8
# # HF_MODEL_PATH = "/capstor/scratch/cscs/jkirchen/singleshot-root/singleshot/outputs/daint_prod_ift_q3-4b/daint_prod_ift_q3-4b_1N4n_ccfee559/step-00000000" 
# # HF_MODEL_PATH = "/capstor/scratch/cscs/jkirchen/singleshot-root/singleshot/outputs/daint_prod_ift_q3-4b/daint_prod_ift_q3-4b_1N4n_ccfee559/step-00030000" 
# HF_MODEL_PATH = "/capstor/scratch/cscs/jkirchen/singleshot-root/singleshot/outputs/daint_prod_ift_q3-4b/daint_prod_ift_q3-4b_1N4n_ccfee559/step-00103476" 
# MODEL_CLASS = "qwen3"
# MASK_ID = 151669
# # EOS_ID = 151645
# EOS_ID = 151643
# # EOS_ID = 48


# # hub based version, make sure to have pushed an updated set of model files!
# HF_MODEL_PATH = "jwkirchenbauer/daint_prod_ift_mask_fix_1N4n_9d30cad5_step-00100160"
# MASK_ID = 128259 # mtp token
# EOS_ID = 128001

HF_MODEL_PATH = "jwkirchenbauer/daint_prod_ift_q3-4b_1N4n_16cdce0f_step-00100160"
MASK_ID = 151669 # mtp token
EOS_ID = 151645 # post trained end message
# EOS_ID = 151643 # pre post-training eos sometimes comes out instead

# Whether to use the base AutoModel classed included in transformers
# which also allows for the hub based remote code version too.
USE_BASE_AUTOMODEL = True # when you want to use the file incl with the pushed model
# USE_BASE_AUTOMODEL = False # meant for debuging the ss repo copy of the modeling file

HF_CHECKPOINT_DIR = Path(HF_MODEL_PATH)

# PRECISION = "bf16-true"
PRECISION = "32-true"
# COMPILE = True
COMPILE = False

# from metamath based validation data/wandb runs
# with completion-y starts of CoTs
# PROMPT = f"""\
# Given $g(x) = x^2$ and $f(x) = 2x - 1$, what is the value of $f(g(2))$?

# First, we need to find $g(2)$.
# Since $g(x) = x^2$, we have $g(2) = \
# """
PROMPT = f"""\
Sandy plans to paint one wall in her bedroom. The wall is 9 feet high and 12 feet long. There is a 2-foot by 4-foot area on that wall that she will not have to paint due to the window. How many square feet will she need to paint?

The total area of the wall is\
"""
# PROMPT = f"""\
# Scott, Mary, and Ken have set a goal of raising $4,000 for their local children's cancer hospital. Currently, Mary's collection is five times the amount Ken has, and three times the amount Scott has. If Ken currently has $600, how much have the three exceeded their goal by?

# If Ken currently has $600, then Mary's collection is 5 * $600 = $3000.
# And Scott's collection is\

# no completion-y training wheels
# PROMPT = f"""\
# Sandy plans to paint one wall in her bedroom. The wall is 9 feet high and 12 feet long. There is a 2-foot by 4-foot area on that wall that she will not have to paint due to the window. How many square feet will she need to paint?\
# """
# PROMPT = f"""\
# Given $g(x) = x^2$ and $f(x) = 2x - 1$, what is the value of $f(g(2))$?\
# """

# PROMPT = f"""\
# <|begin_of_text|>Write a story of exactly 2 paragraphs about a man who wakes up one day and realizes that he's inside a video game. Separate the paragraphs with the markdown divider: ***\n\n \
# """
# PROMPT = f"""\
# <|begin_of_text|>I'm going on a camping trip and I want to make sure I have the best equipment possible. What are some must-haves for a beginner?\n\n \
# """
# PROMPT = f"""\
# <|begin_of_text|>Come up with a special pattern of numbers, and then start repeating it.\n\n \
# """

# NOTE, if you want just a BOS, add it explicitly like above

# APPLY_CHAT_TEMPLATE = False
APPLY_CHAT_TEMPLATE = True

# MAX_NEW_TOKENS = 64
MAX_NEW_TOKENS = 256

# hf only control
DO_MTP = True
# DO_MTP = False

# controls both
# K_TOKS=1
# K_TOKS=3
# K_TOKS=8
K_TOKS=16

STRATEGY = None
STRATEGY = ("conf_adapt", 0.9) # near k=1 perf, "semi-lossless"
# STRATEGY = ("conf_adapt", 0.6) # near k=3 perf "lossy"

# Not well explored/clear purpose for these yet
# STRATEGY = ("random", [1/K_TOKS for _ in range(K_TOKS)]) # uniform
# STRATEGY = ("random", [0.5]+[0 for _ in range(K_TOKS-2)]+[0.5]) # 1 and K_TOKS bimodal
# STRATEGY = ("random", [1/K_TOKS]+[0 for _ in range(K_TOKS-2)]+[(K_TOKS-1)/K_TOKS]) # 1 and K_TOKS bimodal
# STRATEGY = ("conf_adapt_sample@1", 0.9, 0.1) # we include a temperature here

# only applies if using the experimental schemes above with some randomness
SEED = 1234
# SEED = 4321
# SEED = 1823
# SEED = 9669

## HF based generation setup

In [34]:
%autoreload 2

import random
import numpy as np
import torch

from transformers import AutoModelForCausalLM
from transformers import GenerationConfig

try:
    from litgpt.transformers_local.llama.modeling_llama import LlamaForCausalLM
    from litgpt.transformers_local.qwen3.modeling_qwen3 import Qwen3ForCausalLM
    local_import_success = True
except ImportError as e:
    print("Could not import local modeling files, make sure the singleshot repo is in PYTHONPATH if you want to use them.", file=sys.stderr)
    local_import_success = False

def seed_everything_hf(seed):
    # python RNG
    random.seed(seed)
    # pytorch RNGs
    torch.manual_seed(seed)
    # torch.backends.cudnn.deterministic = True
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # numpy RNG
    np.random.seed(seed)

@torch.inference_mode()
def load_hf_model(
    checkpoint_dir: Path,
    precision: Optional[str] = None,
    compile: bool = False,
):
    if USE_BASE_AUTOMODEL:
        # hub and standard
        
        model = AutoModelForCausalLM.from_pretrained(
            checkpoint_dir,
            trust_remote_code=True,
            token=os.environ.get("HF_HUB_TOKEN", None),
            dtype={
                "16-true": torch.float16,
                "bf16-true": torch.bfloat16,
                "32-true": torch.float32,
            }[precision],
            attn_implementation="sdpa",
        )
    else:
        assert local_import_success, "Local modeling files could not be imported."
        # local debugging implementation    
        if MODEL_CLASS == "llama":
            class_ = LlamaForCausalLM
        elif MODEL_CLASS == "qwen3":
            class_ = Qwen3ForCausalLM
        else:
            raise ValueError(f"Unknown MODEL_CLASS+{MODEL_CLASS}")
        model = class_.from_pretrained(
            checkpoint_dir,
            dtype={
                "16-true": torch.float16,
                "bf16-true": torch.bfloat16,
                "32-true": torch.float32,
            }[precision],
            attn_implementation="sdpa",
        )

    model.to(torch.device(f"cuda:{HF_MODEL_GPU}" if torch.cuda.is_available() else "cpu"))
    print(f"Model loaded to use attn_implementation: {model.config._attn_implementation} on device {model.device} in dtype {model.dtype}")
    if compile:
        model = torch.compile(model)
    model.eval()
    return model

@torch.inference_mode()
def generate_response_hf(
    prompt: str = "What food do llamas eat?",
    max_new_tokens: int = None,
    model = None, 
    tokenizer = None,
    streamer = None,
) -> None:
    
    if model.device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(model.device)
    
    encoded = tokenizer.encode(prompt, return_tensors="pt").to(model.device)
    prompt_length = encoded.size(1)
    
    max_returned_tokens = prompt_length + max_new_tokens    

    seed_everything_hf(SEED)
    
    t0 = time.perf_counter()
    output_ids = model.generate(
        input_ids=encoded,
        max_new_tokens=max_new_tokens,
        generation_config=GenerationConfig(
            do_sample=False,
            # temperature=0.0,
            # top_p=1.0,
        ),
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
        streamer=streamer,
        # mtp codepath additional args
        do_mtp=DO_MTP,
        max_returned_tokens=max_returned_tokens,
        k_toks=K_TOKS,
        mask_id=MASK_ID,
        eos_id=EOS_ID,
        include_prompt=True,
        strategy=STRATEGY,
        return_mtp_result_dict=True,
    )
    t = time.perf_counter() - t0
    
    # y = output_ids[0, prompt_length:]
    y = output_ids["token_ids"][0, prompt_length:] # required if return_mtp_result_dict=True
    
    # print(f"#### BEGIN Prompt ####")
    # print(f"{prompt}")
    # print(f"########")
    # print(f"tokenized as:\n{encoded.squeeze(0).tolist()}")
    # print(f"#### END Prompt ####")
    # print(f"#### BEGIN Generation ####")
    # print(f"{tokenizer.decode(y, skip_special_tokens=False)}")
    # print(f"########")
    # print(f"tokenized as:\n{y.tolist()}")
    # print(f"#### END Generation ####")
    # tokens_generated = y.size(0)
    # print(
    #     f"Time for inference: {t:.02f} sec total, {tokens_generated / t:.02f} tokens/sec over {tokens_generated} tokens."
    # )
    if model.device.type == "cuda":
        max_memory_allocated_per_gpu = torch.cuda.max_memory_allocated(model.device) / 1024**3
        max_memory_reserved_per_gpu = torch.cuda.max_memory_reserved(model.device) / 1024**3
        torch.cuda.reset_peak_memory_stats(model.device)
        print(
            f"Memory alloc/reserved: {max_memory_allocated_per_gpu:.02f} GB / {max_memory_reserved_per_gpu:.02f} GB",
            flush=True
        )
    return y, encoded, output_ids

In [35]:
hf_tokenizer = AutoTokenizer.from_pretrained(HF_CHECKPOINT_DIR)
streamer = TextStreamer(hf_tokenizer)

hf_model = load_hf_model(
    checkpoint_dir=HF_CHECKPOINT_DIR,
    precision=PRECISION,
    compile=COMPILE,
)

A new version of the following files was downloaded from https://huggingface.co/jwkirchenbauer/daint_prod_ift_q3-4b_1N4n_16cdce0f_step-00100160:
- configuration_qwen3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/jwkirchenbauer/daint_prod_ift_q3-4b_1N4n_16cdce0f_step-00100160:
- modeling_qwen3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
Fetching 3 files: 100%|██████████| 3/3 [00:10<00:00,  3.52s/it]


Loading local Qwen3ForCausalLM!


Loading checkpoint shards: 100%|██████████| 3/3 [00:31<00:00, 10.43s/it]


Model loaded to use attn_implementation: sdpa on device cuda:0 in dtype torch.float32


## HF based generation execute

In [36]:
def format_query_for_chat_generation(query, do_tokenize=False):
        messages = [
            {"role": "user", "content": query},
        ]
        return hf_tokenizer.apply_chat_template(messages, tokenize=do_tokenize, add_generation_prompt=True)
if APPLY_CHAT_TEMPLATE:
    PROMPT = format_query_for_chat_generation(PROMPT, do_tokenize=False)

hf_generation, hf_prompt_encoding, mtp_dict = generate_response_hf(
    prompt=PROMPT, 
    max_new_tokens=MAX_NEW_TOKENS,
    tokenizer=hf_tokenizer,
    model=hf_model,
    streamer=streamer,
)

Routing through local (hub) generate impl!
Executing custom MTP generation codepath!
Stripped these found keys from passed kwargs prior to MTP generation: ['max_new_tokens', 'eos_token_id', 'pad_token_id']
About to call generate_mtp with final args: (), kwargs.keys: dict_keys(['generation_config', 'streamer', 'model', 'prompt']), mtp_kwargs: dict_keys(['max_returned_tokens', 'k_toks', 'mask_id', 'eos_id', 'include_prompt', 'strategy', 'return_mtp_result_dict'])

<BEGIN Streaming Prompt>
<|im_start|>user
Sandy plans to paint one wall in her bedroom. The wall is 9 feet high and 12 feet long. There is a 2-foot by 4-foot area on that wall that she will not have to paint due to the window. How many square feet will she need to paint?

The total area of the wall is<|im_end|>
<|im_start|>assistant

<END Streaming Prompt>

<BEGIN Streaming Generation>
To find how many square feet Sandy will need to paint, let's go step by step.

---

**Step 1: Find the total area of the wall**

**

Wall height

In [37]:
# print(mtp_dict)
raw_token_ids = mtp_dict['token_ids'][0][hf_prompt_encoding.shape[1]:].tolist()
eff_k_arr = mtp_dict['effective_k_values']

def print_chunked_output(token_ids, eff_k_values, tokenizer=hf_tokenizer):
    """eff_k_values is a list/array whose sum is the same length as token_ids, indicating the effective k , or chunk size used at each generation step.
    This prints out the token ids chunked according to the effective k values.
    """
    print(f"Chunked output decoded:")
    idx = 0
    all_chunks = []
    for step, k in enumerate(eff_k_values):
        chunk = token_ids[idx:idx+k]
        decoded_chunk = tokenizer.decode(chunk, skip_special_tokens=False)
        all_chunks.append(f"step {step} @ k={k} | {decoded_chunk}")
        idx += k
    print("\n".join(all_chunks))
    
print_chunked_output(raw_token_ids, eff_k_arr)

Chunked output decoded:
step 0 @ k=1 | To
step 1 @ k=2 |  find how
step 2 @ k=4 |  many square feet Sandy
step 3 @ k=1 |  will
step 4 @ k=4 |  need to paint,
step 5 @ k=1 |  let
step 6 @ k=2 | 's go
step 7 @ k=1 |  step
step 8 @ k=2 |  by step
step 9 @ k=2 | .

---


step 10 @ k=1 | **
step 11 @ k=1 | Step
step 12 @ k=2 |  1
step 13 @ k=2 | : Find
step 14 @ k=8 |  the total area of the wall**

**


step 15 @ k=1 | Wall
step 16 @ k=1 |  height
step 17 @ k=4 |  = 9 feet
step 18 @ k=2 |   
  

step 19 @ k=2 | Wall length
step 20 @ k=5 |  = 12 feet
step 21 @ k=1 |   


step 22 @ k=1 | So
step 23 @ k=1 | ,
step 24 @ k=2 |  total area
step 25 @ k=1 |  =
step 26 @ k=1 |  height
step 27 @ k=1 |  ×
step 28 @ k=1 |  length
step 29 @ k=1 |   

step 30 @ k=1 | =
step 31 @ k=3 |  9 ×
step 32 @ k=4 |  12  

step 33 @ k=1 | =
step 34 @ k=3 |  **10
step 35 @ k=4 | 8 square feet**


step 36 @ k=1 | ---


step 37 @ k=3 | **Step 
step 38 @ k=3 | 2: Find
step 39 @ k=3 |  the area of
step 40 @ k=2 |  the w

## Special token debugging/ID-ifying

In [38]:
from transformers import AutoTokenizer

TOK_PATH = HF_CHECKPOINT_DIR

# TOK_PATH = "checkpoints/extended/Magpie-Align/Llama-3.1-8B-Magpie-Align-SFT-v0.1-MTPV128384"
# TOK_PATH = "checkpoints/extended/Qwen/Qwen3-4B-Instruct-2507-MTP"

tokenizer = AutoTokenizer.from_pretrained(TOK_PATH)

In [39]:

def format_chat(example, do_tokenize=False):
        messages = [
            {"role": "user", "content": example["query"]},
            {"role": "assistant", "content": example["response"]},
        ]
        return {
            "text": tokenizer.apply_chat_template(
                messages, tokenize=do_tokenize
            )
        }

example = {
    "query": "Hello world!",
    "response": "Hi there! How can I assist you today?"
}
formatted = format_chat(example, do_tokenize=False)
print(formatted["text"])

<|im_start|>user
Hello world!<|im_end|>
<|im_start|>assistant
Hi there! How can I assist you today?<|im_end|>



In [41]:
# TGT_STR = ""
# TGT_STR = "<|eot_id|><|start_header_id|>user<|end_header_id|>"
# TGT_STR = "Q: There are 15 trees in the grove."
# TGT_STR = "A: There are 15 trees in the grove."
# TGT_STR = "Q:"
# TGT_STR = "<|endoftext|>"
TGT_STR = "<|im_end|>"

"-".join([str(i) for i  in tokenizer.encode(TGT_STR, add_special_tokens=False)])

'151645'